# 🚀 Leveraging Apache Spark for Scalable Retail Data Processing

## 📌 Overview
In this project, we simulate a real-world big data scenario by processing large-scale retail data using **Apache Spark**.

The goal is to demonstrate how distributed data processing can efficiently handle high-volume datasets and generate actionable business insights.

💡 This project reflects a typical Data Engineering workflow — from raw data ingestion to transformation, aggregation, and insight generation.

---

## 🎯 Objectives
- Load and process retail data using **Spark RDDs**
- Perform data cleaning and transformations
- Apply aggregations to analyze key business metrics
- Evaluate sales, revenue, pricing, and promotion performance
- Demonstrate scalable distributed data processing

---

## 🛠️ Technologies Used
- **Apache Spark (RDD API)**
- **Python (PySpark)**

---

## 📊 Dataset
The dataset represents retail sales transactions and includes:

- `product_id` – unique product identifier  
- `store_id` – store identifier  
- `date` – transaction date  
- `sales` – number of units sold  
- `revenue` – total revenue generated  
- `stock` – available inventory  
- `price` – product price  
- `promotion types` – applied marketing campaigns  

💡 The dataset contains **1,033,435 records**, making it suitable for large-scale distributed data processing with Apache Spark. 

---

## 🧩 Business Problem
RetailWorld needs to efficiently process **large volumes of sales data**.

Traditional tools struggle to scale, resulting in:
- Slow data processing  
- Delayed reporting  
- Limited ability to make timely decisions  

---

## ⚡ Solution
To address these challenges, we use **Apache Spark** to:

- Process data in a distributed manner  
- Perform fast transformations and aggregations  
- Enable scalable analytics on large datasets  

💡 This approach enables businesses to make **faster, data-driven decisions at scale**.

In [1]:
# !pip install pyspark
# !pip install findspark

### 1. Initialize Spark

We initialize a Spark session to enable distributed data processing.

💡 SparkSession is the entry point for working with Spark and allows us to access the underlying SparkContext.

In [60]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder \
    .appName("RetailStoreSalesAnalysis") \
    .getOrCreate()

# Get SparkContext
sc = spark.sparkContext

print(sc)


<SparkContext master=local[*] appName=RetailStoreSalesAnalysis>


### 2. Download Dataset

We download the retail dataset from a public cloud storage.

💡 This dataset will be used for large-scale data processing with Spark.

In [61]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/XXlNzqYcxqkTbllc-tL_0w/Retailsales.csv

--2026-05-03 13:02:11--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/XXlNzqYcxqkTbllc-tL_0w/Retailsales.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 47593992 (45M) [text/csv]
Saving to: ‘Retailsales.csv’

Retailsales.csv     100%[===================>]  45.39M  42.1MB/s    in 1.1s    

2026-05-03 13:02:14 (42.1 MB/s) - ‘Retailsales.csv’ saved [47593992/47593992]



Explanation:

The dataset is loaded into a Spark RDD to enable distributed processing of large-scale retail data.

### 3. Load and Inspect Data

After loading the dataset, we inspect its structure and size to understand the scale of data being processed.

💡 The dataset contains over 1 million records, making it suitable for distributed processing with Apache Spark.

In [63]:
# Load dataset into RDD
rdd_raw = sc.textFile("Retailsales.csv")

# Preview data
rdd_raw.take(5)

['product_id,store_id,date,sales,revenue,stock,price,promo_type_1,promo_bin_1,promo_type_2',
 'P0001,S0002,1/2/2017,0,0,8,6.25,PR14,,PR03',
 'P0001,S0012,1/2/2017,1,5.3,0,6.25,PR14,,PR03',
 'P0001,S0013,1/2/2017,2,10.59,0,6.25,PR14,,PR03',
 'P0001,S0023,1/2/2017,0,0,6,6.25,PR14,,PR03']

In [65]:
# Count total number of records
total_records = rdd_raw.count()

print(f"Total number of records: {total_records:,}")

[Stage 3:>                                                          (0 + 2) / 2]

Total number of records: 1,033,435


💡 The dataset contains over 1 million records, confirming its suitability for large-scale data processing.

### 4. Remove Header

We remove the header row to process only actual data records.

💡 This ensures that the header is not treated as a data entry.

In [66]:
# Extract header row
header = rdd_raw.first()

# Remove header from dataset
rdd_data = rdd_raw.filter(lambda row: row != header)

### 5. Parse Data
In this step, we split each raw text record into structured columns using a comma delimiter.

Since the dataset is initially loaded as plain text, each row must be parsed into individual fields to enable further processing and analysis.

In [67]:
# Split rows into columns
rdd_split = rdd_data.map(lambda row: row.split(","))

# Preview parsed data
rdd_split.take(5)

[['P0001', 'S0002', '1/2/2017', '0', '0', '8', '6.25', 'PR14', '', 'PR03'],
 ['P0001', 'S0012', '1/2/2017', '1', '5.3', '0', '6.25', 'PR14', '', 'PR03'],
 ['P0001', 'S0013', '1/2/2017', '2', '10.59', '0', '6.25', 'PR14', '', 'PR03'],
 ['P0001', 'S0023', '1/2/2017', '0', '0', '6', '6.25', 'PR14', '', 'PR03'],
 ['P0001', 'S0025', '1/2/2017', '0', '0', '1', '6.25', 'PR14', '', 'PR03']]

### 6. Parsing and Cleaning Data

In this step, we convert the parsed data into a structured and typed format suitable for analysis.

Each column is mapped to its appropriate data type (e.g., numerical fields such as sales, revenue, stock, and price are cast to float), ensuring accurate computations.

We also perform basic data cleaning by filtering out invalid records where key metrics (such as sales and price) are less than or equal to zero.

In [68]:
# Convert and clean data
rdd_parsed = rdd_split.map(lambda x: (
    x[0],                 # product_id
    x[1],                 # store_id
    x[2],                 # date
    float(x[3]),          # sales
    float(x[4]),          # revenue
    float(x[5]),          # stock
    float(x[6]),          # price
    x[7],                 # promo_type_1
    x[8],                 # promo_bin_1
    x[9]                  # promo_type_2
))

# Remove invalid records (sales and price > 0)
rdd_clean = rdd_parsed.filter(lambda x: x[3] > 0 and x[6] > 0)

# Preview cleaned data
rdd_clean.take(5)

[('P0001', 'S0012', '1/2/2017', 1.0, 5.3, 0.0, 6.25, 'PR14', '', 'PR03'),
 ('P0001', 'S0013', '1/2/2017', 2.0, 10.59, 0.0, 6.25, 'PR14', '', 'PR03'),
 ('P0001', 'S0056', '1/2/2017', 1.0, 5.3, 6.0, 6.25, 'PR14', '', 'PR03'),
 ('P0001', 'S0103', '1/2/2017', 1.0, 5.3, 10.0, 6.25, 'PR14', '', 'PR03'),
 ('P0001', 'S0106', '1/2/2017', 1.0, 5.3, 3.0, 6.25, 'PR14', '', 'PR03')]

💡 Invalid records (e.g., with zero or negative sales and price) were removed to improve data quality.

### 7. Data After Cleaning

After removing invalid or unnecessary records, we evaluate the impact of data cleaning on the dataset size.

This step helps ensure that only meaningful and reliable data is used for further analysis.

In [70]:
# Count records after cleaning
cleaned_count = rdd_clean.count()

print(f"Number of records after cleaning: {cleaned_count:,}")

[Stage 8:>                                                          (0 + 2) / 2]

Number of records after cleaning: 196,661


📊 The dataset was reduced after cleaning, indicating removal of invalid or irrelevant records.

In [71]:
# Compare before and after cleaning
print(f"Original records: {total_records:,}")
print(f"After cleaning: {cleaned_count:,}")

Original records: 1,033,435
After cleaning: 196,661


💡 The dataset was reduced from **1,033,435 to 196,661 records**, meaning that a significant portion of the data (~80%) was filtered out due to invalid or low-quality values.

This highlights the importance of data cleaning in real-world pipelines, where raw data often contains noise, errors, or irrelevant records that can negatively impact analysis results.

### 8. Check Partitions

In this step, we analyze how the dataset is distributed across Spark partitions.

Proper partitioning is critical for efficient parallel processing, as it directly impacts performance and scalability in distributed systems.

In [72]:
# Check number of partitions
print(f"Number of partitions: {rdd_clean.getNumPartitions()}")

Number of partitions: 2


💡 The dataset is distributed across **2 partitions**, which allows Spark to process data in parallel.

For larger datasets, increasing the number of partitions can improve performance by enabling better workload distribution across cluster resources.

### 9. Partition-wise Record Count

This step analyzes how records are distributed across Spark partitions.

Balanced partitions are essential for efficient parallel processing and optimal resource utilization.

In [73]:
# Count the number of records in each partition
def count_in_partition(index, iterator):
    count = sum(1 for _ in iterator)
    yield (index, count)

# Apply the function to each partition
partition_counts = rdd_clean.mapPartitionsWithIndex(count_in_partition).collect()

# Print partition-wise record counts
print("Number of records in each partition:")
for partition, count in partition_counts:
    print(f"Partition {partition}: {count} records")

[Stage 9:>                                                          (0 + 2) / 2]

Number of records in each partition:
Partition 0: 97534 records
Partition 1: 99127 records


💡 The records are distributed almost evenly across partitions (97,534 vs 99,127 records), indicating a well-balanced workload.

This suggests that the data processing will be efficient, as no single partition becomes a bottleneck during computation.

### 10. Aggregations

### a. Total Sales and Revenue per Product

This step computes the **total sales** and **total revenue** for each product.

Each record is transformed into a key-value pair:

- **Key:** `product_id`  
- **Value:** `(sales, revenue)`

The `reduceByKey()` function is used to aggregate (sum) values for each product.

💡 This allows us to identify top-performing products and analyze revenue distribution across the dataset.

In [74]:
# Calculate total sales and revenue per product
sales_revenue_per_product = rdd_clean.map(lambda x: (x[0], (x[3], x[4]))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .mapValues(lambda x: (round(x[0], 2), round(x[1], 2)))

# Preview results
sales_revenue_per_product.take(10)

[('P0016', (60.0, 111.08)),
 ('P0051', (26381.0, 16782.56)),
 ('P0055', (1136.0, 3650.86)),
 ('P0057', (233.0, 2478.77)),
 ('P0067', (286.0, 3799.19)),
 ('P0068', (255.0, 1188.18)),
 ('P0070', (1366.0, 6895.11)),
 ('P0071', (63.0, 484.85)),
 ('P0079', (5464.0, 10949.09)),
 ('P0097', (138.0, 1506.68))]

💡 The results show that some products significantly outperform others.

For example, product **P0051** has much higher sales and revenue compared to most products, indicating strong demand or effective pricing/marketing.

This helps identify top-performing products for further business optimization.

### b. Total Sales and Revenue per Store

This step computes the **total sales** and **total revenue** for each store.

Each record is transformed into a key-value pair:
- **Key:** `store_id`
- **Value:** `(sales, revenue)`

The `reduceByKey()` function is then applied to aggregate (sum) the values for each store.

💡 This aggregation helps evaluate store performance, compare locations, and identify top-performing stores driving the most revenue.

In [75]:
# Calculate total sales and revenue per store
sales_revenue_per_store = rdd_clean.map(lambda x: (x[1], (x[3], x[4]))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .mapValues(lambda x: (round(x[0], 2), round(x[1], 2)))

# Preview results
sales_revenue_per_store.take(10)

[('S0013', (3969.6, 16489.44)),
 ('S0106', (5526.99, 13176.46)),
 ('S0044', (1378.0, 2432.52)),
 ('S0001', (6375.7, 27373.84)),
 ('S0032', (2655.77, 5713.52)),
 ('S0040', (9843.92, 33434.98)),
 ('S0068', (1387.01, 3096.19)),
 ('S0085', (37937.73, 117882.5)),
 ('S0015', (6409.68, 24111.38)),
 ('S0020', (18188.02, 69049.52))]

💡 The results show significant differences in store performance.

For example, store **S0085** generates much higher revenue compared to stores like **S0044**, indicating strong variation in sales performance across locations.

This helps identify top-performing stores and those that may require improvement.

### c. Average Price per Product

This step calculates the **average price** for each product.

Each record is transformed into a key-value pair:
- **Key:** `product_id`
- **Value:** `(price, 1)`

The `reduceByKey()` function is used to compute the total price and count per product.  
Then, the average price is calculated by dividing the total price by the count.

💡 This aggregation helps analyze pricing patterns, detect inconsistencies, and compare product-level pricing strategies across the dataset.

In [76]:
# Calculate total price and count per product
price_count_per_product = rdd_clean.map(lambda x: (x[0], (x[6], 1))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Calculate average price
average_price_per_product = price_count_per_product.mapValues(
    lambda x: round(x[0] / x[1], 2)
)

# Preview results
average_price_per_product.take(10)

[('P0016', 2.0),
 ('P0051', 0.7),
 ('P0055', 3.49),
 ('P0057', 13.92),
 ('P0067', 16.37),
 ('P0068', 5.31),
 ('P0070', 6.22),
 ('P0071', 9.27),
 ('P0079', 2.24),
 ('P0097', 11.96)]

💡 The results show significant variation in product pricing.

For example, product **P0067 has a high average price (16.37)**, while **P0051 has a very low average price (0.70)**.

This indicates a mix of pricing strategies:
- low-price, high-volume products
- high-price, lower-volume products

Such insights can help optimize pricing and positioning strategies.

### d. Sales and Revenue per Promotion Type

#### promo_type_1

This step calculates the **total sales** and **total revenue** for each promotion type.

Each record is transformed into a key-value pair:
- **Key:** `promo_type_1`
- **Value:** `(sales, revenue)`

The `reduceByKey()` function aggregates the values for each promotion category.

💡 This analysis helps evaluate the effectiveness of different promotion strategies, identify high-performing campaigns, and measure their impact on overall sales and revenue.

In [77]:
# Aggregation for promo_type_1
sales_revenue_per_promo_1 = rdd_clean.map(lambda x: (x[7], (x[3], x[4]))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .mapValues(lambda x: (round(x[0], 2), round(x[1], 2)))

# Preview results
sales_revenue_per_promo_1.take(10)

[('PR05', (13645.0, 81005.96)),
 ('PR03', (7957.81, 53616.67)),
 ('PR06', (3800.0, 7620.63)),
 ('PR07', (838.0, 4939.66)),
 ('PR12', (4319.0, 20421.66)),
 ('PR08', (1007.0, 12547.35)),
 ('PR09', (3233.0, 13896.21)),
 ('PR14', (547741.66, 1482423.33)),
 ('PR10', (19971.0, 66575.17)),
 ('PR17', (298.0, 2198.06))]

💡 The results clearly show that promotion effectiveness varies significantly.

For example, promotion **PR14 generated extremely high results (over 547,741 sales and 148,423 revenue)**, significantly outperforming all other campaigns.

In contrast, promotions like **PR07 generated less than 1,000 sales**, indicating much lower impact.

This suggests that certain promotions are far more effective and should be prioritized in marketing strategies.

#### promo_type_2

This step calculates the **total sales** and **total revenue** for the second promotion type.

Each record is transformed into a key-value pair:
- **Key:** `promo_type_2`
- **Value:** `(sales, revenue)`

The `reduceByKey()` function aggregates sales and revenue for each promotion category.

💡 In this dataset, `promo_type_2` contains only a single category, resulting in a single aggregated record after grouping. This demonstrates how aggregation behaves when the key has low cardinality.

In [78]:
# Aggregation for promo_type_2
sales_revenue_per_promo_2 = rdd_clean.map(lambda x: (x[9], (x[3], x[4]))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .mapValues(lambda x: (round(x[0], 2), round(x[1], 2)))

# Preview results
sales_revenue_per_promo_2.take(10)

[('PR03', (612642.47, 1968846.77))]

💡 All records belong to a single promotion (PR03), which generated very high sales and revenue.

This indicates that promo_type_2 has only one promotion type.

### 11. Stock Analysis per Product

This step calculates the **total stock available** for each product across all stores.

Each record is transformed into a key-value pair:
- **Key:** `product_id`
- **Value:** `stock`

The `reduceByKey()` function is used to aggregate (sum) stock values for each product.

💡 This analysis helps understand inventory levels and identify products with high or low stock availability.

In [79]:
# Calculate total stock per product
stock_per_product = rdd_clean.map(lambda x: (x[0], x[5])) \
    .reduceByKey(lambda a, b: a + b)

# Preview results
stock_per_product.take(10)

[('P0016', 1032.0),
 ('P0051', 476816.0),
 ('P0055', 14777.0),
 ('P0057', 1317.0),
 ('P0067', 1501.0),
 ('P0068', 1970.0),
 ('P0070', 6191.0),
 ('P0071', 225.0),
 ('P0079', 161374.0),
 ('P0097', 2041.0)]

💡 Stock levels vary significantly across products.

For example, **P0051 has very high stock**, while **P0071 has very low stock**.

This may indicate overstocking for some products and risk of stock shortages for others.

### 12. Saving Results

The aggregation results can be saved for further analysis and reuse.

In production, these outputs could be written to HDFS, cloud storage, or a data warehouse.

💡 In this portfolio notebook, results are displayed directly to keep the repository lightweight and easy to review.


In [24]:
# Optional: save results locally as text files
# Note: Spark saveAsTextFile requires the output path not to exist.

# sales_revenue_per_product.saveAsTextFile("output/sales_revenue_per_product")
# sales_revenue_per_store.saveAsTextFile("output/sales_revenue_per_store")
# average_price_per_product.saveAsTextFile("output/average_price_per_product")
# sales_revenue_per_promo_1.saveAsTextFile("output/sales_revenue_per_promo_1")
# sales_revenue_per_promo_2.saveAsTextFile("output/sales_revenue_per_promo_2")
# stock_per_product.saveAsTextFile("output/stock_per_product")

📁 For local development, results are also exported as CSV files 
to enable easy inspection and analysis outside Spark.

In [80]:
# Save results as CSV (local development)

sales_revenue_per_product.toDF(["product_id", "sales_revenue"]) \
    .toPandas().to_csv("output/sales_revenue_per_product.csv", index=False)

sales_revenue_per_store.toDF(["store_id", "sales_revenue"]) \
    .toPandas().to_csv("output/sales_revenue_per_store.csv", index=False)

average_price_per_product.toDF(["product_id", "avg_price"]) \
    .toPandas().to_csv("output/average_price_per_product.csv", index=False)

stock_per_product.toDF(["product_id", "stock"]) \
    .toPandas().to_csv("output/stock_per_product.csv", index=False)

print("Results saved as CSV")

Results saved as CSV


### 13. Printing Results

In this step, we display a sample of the results from each aggregation.

Instead of collecting the entire dataset, we use `.take(10)` to retrieve a small subset of records.

💡 This approach is more efficient and prevents memory issues when working with large datasets.

In [81]:
# Print results (sample only)

print("Total Sales and Revenue per Product:")
print("=" * 35)
print("ID     | Sales      | Revenue")
print("-" * 35)
for product in sales_revenue_per_product.take(10):
    print(f"{product[0]:<6} | {product[1][0]:<10.2f} | {product[1][1]:<10.2f}")


print("\nTotal Sales and Revenue per Store:")
print("=" * 35)
print("Store  | Sales      | Revenue")
print("-" * 35)
for store in sales_revenue_per_store.take(10):
    print(f"{store[0]:<6} | {store[1][0]:<10.2f} | {store[1][1]:<10.2f}")


print("\nAverage Price per Product:")
print("=" * 30)
print("ID     | Avg Price")
print("-" * 30)
for product in average_price_per_product.take(10):
    print(f"{product[0]:<6} | {product[1]:<10.2f}")


print("\nSales and Revenue per Promotion Type 1:")
print("=" * 40)
print("Promo  | Sales      | Revenue")
print("-" * 40)
for promo in sales_revenue_per_promo_1.take(10):
    print(f"{promo[0]:<6} | {promo[1][0]:<10.2f} | {promo[1][1]:<10.2f}")


print("\nSales and Revenue per Promotion Type 2:")
print("=" * 40)
print("Promo  | Sales      | Revenue")
print("-" * 40)
for promo in sales_revenue_per_promo_2.take(10):
    print(f"{promo[0]:<6} | {promo[1][0]:<10.2f} | {promo[1][1]:<10.2f}")


print("\nStock per Product:")
print("=" * 25)
print("ID     | Stock")
print("-" * 25)
for product in stock_per_product.take(10):
    print(f"{product[0]:<6} | {product[1]:<10.2f}")

Total Sales and Revenue per Product:
ID     | Sales      | Revenue
-----------------------------------
P0016  | 60.00      | 111.08    
P0051  | 26381.00   | 16782.56  
P0055  | 1136.00    | 3650.86   
P0057  | 233.00     | 2478.77   
P0067  | 286.00     | 3799.19   
P0068  | 255.00     | 1188.18   
P0070  | 1366.00    | 6895.11   
P0071  | 63.00      | 484.85    
P0079  | 5464.00    | 10949.09  
P0097  | 138.00     | 1506.68   

Total Sales and Revenue per Store:
Store  | Sales      | Revenue
-----------------------------------
S0013  | 3969.60    | 16489.44  
S0106  | 5526.99    | 13176.46  
S0044  | 1378.00    | 2432.52   
S0001  | 6375.70    | 27373.84  
S0032  | 2655.77    | 5713.52   
S0040  | 9843.92    | 33434.98  
S0068  | 1387.01    | 3096.19   
S0085  | 37937.73   | 117882.50 
S0015  | 6409.68    | 24111.38  
S0020  | 18188.02   | 69049.52  

Average Price per Product:
ID     | Avg Price
------------------------------
P0016  | 2.00      
P0051  | 0.70      
P0055  | 3.49    

### 14. Cleanup

This step ensures that all allocated resources are released and the Spark application shuts down cleanly.

💡 Proper resource management is essential when working with distributed systems like Apache Spark, especially in production environments.

In [82]:
# Stop Spark context to release resources
sc.stop()

### 15. Business Insights

Based on the analysis of retail sales data using Apache Spark, several key business insights can be derived:

### 📦 Product Performance
- A small number of products generate a disproportionately large share of total revenue.
- For example, product **P0051 generated over 26,381 sales and 16,782 revenue**, significantly outperforming other products.
- In contrast, many products (e.g., P0016 with only 60 sales) contribute minimally to overall performance.
- High-performing products can be prioritized for marketing, promotions, and inventory allocation.
- Low-performing products may require reassessment or optimization strategies.

### 🏬 Store Performance
- Sales and revenue vary significantly across stores.
- For instance, store **S0085 generated over 117,882 revenue**, while store S0044 generated only around 2,432 revenue.
- This indicates strong performance differences between locations.
- High-performing stores may benefit from favorable location, stronger demand, or effective local strategies.
- Underperforming stores may require operational improvements.

### 💰 Pricing Strategy
- Average product prices vary considerably across the catalog.
- For example, product **P0067 has a high average price of 16.37**, while **P0051 has a low average price of 0.70**, yet still achieves very high sales volume.
- This suggests a combination of:
  - Volume-driven products (low price, high sales)
  - Margin-driven products (high price, lower volume)
- Pricing strategies can be optimized based on product performance and demand.

### 🎯 Promotion Effectiveness
- Different promotion types have a measurable impact on sales and revenue.
- Promotion **PR14 shows extremely high performance with over 547,741 sales and 148,423 revenue**, significantly outperforming other campaigns.
- In comparison, promotions like PR07 generate less than 1,000 sales.
- This indicates that certain promotions are far more effective and should be prioritized.
- These insights can be used to:
  - Optimize marketing campaigns
  - Focus on the most effective promotion strategies
  
### 📊 Inventory (Stock) Insights
- Stock levels vary significantly across products.
- For example, product **P0051 has extremely high stock levels (476,816 units)**, which may indicate overstocking.
- On the other hand, product **P0071 has very low stock (225 units)**, which may lead to stockouts.
- Inventory can be optimized by aligning stock levels with demand patterns and sales performance.

---

## 🚀 Conclusion

This project demonstrates how Apache Spark can be used to efficiently process and analyze large-scale retail data.

By leveraging distributed computing, businesses can:
- Identify top-performing products and stores (e.g., P0051, S0085)
- Optimize pricing and promotional strategies (e.g., PR14 effectiveness)
- Improve inventory management by balancing stock levels
- Make faster, data-driven decisions at scale

💡 Overall, data engineering plays a critical role in enabling scalable analytics and supporting business intelligence.